In [1]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
file = "C:/Users/jlevi/Desktop/UNI/ITNPA4- 14/research_abstracts.json"

df = pd.read_json(file)

df

,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical
...,...,...,...
1133,10.1007/bf00123887,Concluding remarksWhile the final version of t...,classical
1134,10.1093/acrefore/9780190625979.013.171,High-Dimensional Dynamic Factor Models have th...,classical
1135,10.1287/mnsc.2013.1812,"Francesca Gino, Erin L. Krupka, Roberto A. Web...",classical
1136,10.2118/22022-pa,"Summary EOR projects, notably CO2 floods, are ...",classical


In [3]:
#check for duplicate entries
duplicates = df[df.duplicated()]

print(duplicates)

#drop duplicates

df = df.drop_duplicates()

duplicated_sum = df.duplicated().sum()
print(duplicated_sum)

       DOI                                           Abstract Label
1040  None  Inflation is a major determinant for allocatio...    ai
1041  None  This paper proposes a parsimoniously time vary...    ai
0


In [4]:
df.head()


,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical


In [5]:
#check for missing values

print(df.isnull().sum())

df = df.dropna()

print(df.isnull().sum())

DOI         2
Abstract    0
Label       0
dtype: int64
DOI         0
Abstract    0
Label       0
dtype: int64


In [6]:
# dropping uneeded column

df = df.drop(columns =['DOI'])

df.head()

,Abstract,Label
0,Forward-looking forecasting of the inflation r...,ai
1,Our study examined the disaggregation of infla...,ai
2,This paper develops a dynamic factor model tha...,classical
3,"In this paper, I use a simple SIR Macro model ...",classical
4,BackgroundGame theory is useful for identifyin...,classical


In [7]:
#remove stopwords

stop_words = set(stopwords.words("english"))

df['Abstract']=df['Abstract'].apply( lambda x: " ".join(word for word in x.split() if word not in stop_words))


In [8]:
#make info lowercase
df['Abstract']=df['Abstract'].str.lower()

#remove punctuation 
df['Abstract'] =df['Abstract'].apply(lambda x: re.sub(r'[^\w\s]','',x))

In [9]:
#lemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

df["Abstract"] = df["Abstract"].apply(
    lambda x: " ".join(lemmatizer.lemmatize(word) for word in x.split())
)

In [10]:
#Feature coloumn

X = df['Abstract']

#target coloumn
y= df['Label']

print(X.head())
print(y.head())

0    forwardlooking forecasting inflation rate coul...
1    our study examined disaggregation inflation co...
2    this paper develops dynamic factor model us eu...
3    in paper i use simple sir macro model examine ...
4    backgroundgame theory useful identifying condi...
Name: Abstract, dtype: object
0           ai
1           ai
2    classical
3    classical
4    classical
Name: Label, dtype: object


In [11]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts(X)

X_sequences = tokenizer.texts_to_sequences(X)

In [12]:
#make sequences the same length
X_padded = pad_sequences(X_sequences)

In [13]:
#convert labels to numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

#split into training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [14]:
model = Sequential()

model.add(Embedding(
    input_dim=len(tokenizer.word_index) + 1,
    output_dim=100))

model.add(SimpleRNN(64))

model.add(Dense(1, activation="sigmoid"))

In [15]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 30s 806ms/step - accuracy: 0.7293 - loss: 0.5685 - val_accuracy: 0.8106 - val_loss: 0.4933
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 22s 756ms/step - accuracy: 0.7895 - loss: 0.4902 - val_accuracy: 0.8106 - val_loss: 0.4953
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 25s 869ms/step - accuracy: 0.8545 - loss: 0.3411 - val_accuracy: 0.7797 - val_loss: 0.5179
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 23s 802ms/step - accuracy: 0.9106 - loss: 0.2182 - val_accuracy: 0.7709 - val_loss: 0.5130
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 39s 727ms/step - accuracy: 1.0000 - loss: 0.0576 - val_accuracy: 0.7709 - val_loss: 0.5292
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 22s 758ms/step - accuracy: 1.0000 - loss: 0.0247 - val_accuracy: 0.7489 - val_loss: 0.5447
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 22s 763ms/step - accuracy: 1.0000 - loss: 0.0149 - val_accuracy: 0.7357 - val_loss: 0.5568
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 23s 772ms/step - accuracy: 1.0000 - loss: 0.0105 - val_accu

In [17]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.7147 - loss: 0.5552
Test Accuracy: 0.7136563658714294


In [18]:
from sklearn.metrics import confusion_matrix

predictions = model.predict(X_test)
predicted_classes = (predictions > 0.5).astype(int)

cm = confusion_matrix(y_test, predicted_classes)

print(cm)

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 221ms/step
[[ 17  26]
 [ 39 145]]
